# 🔧 Notebook 01: MLX Fundamentals

Welcome to the MLX deep-dive. This notebook covers the core building blocks of Apple's ML framework — arrays, lazy evaluation, automatic differentiation, neural network modules, and the training loop pattern. Everything here runs natively on Metal, no CUDA required.

**What you'll learn:**
1. MLX arrays and dtypes
2. Lazy evaluation and kernel fusion
3. Automatic differentiation with `mx.grad`
4. Neural network modules (`mlx.nn`)
5. The MLX training loop pattern

**Prerequisites:** Notebook 00 (Environment & Apple Silicon)

## ✅ Environment Validation

Before we start, let's verify that MLX, Metal, and Python are set up correctly. This cell uses our shared `utils.checks` module to run all the checks at once.

In [ ]:
from utils.checks import validate_environment, print_environment_report

results = print_environment_report()

# Bail out early if MLX isn't available
assert results["mlx_available"], "MLX is required — run: pip install mlx"
assert results["metal_available"], "Metal GPU not found. Check macOS version and hardware."

print("\n🟢 All checks passed — ready to learn MLX!")

---
## 🆚 MLX vs PyTorch: Key Differences

Before writing any code, let's understand what makes MLX different from PyTorch. MLX was designed from the ground up for Apple Silicon, and several design choices reflect that.

| Feature | MLX | PyTorch |
|---|---|---|
| **Evaluation** | Lazy (operations recorded, executed on `mx.eval()`) | Eager (operations execute immediately) |
| **GPU Backend** | Metal (Apple Silicon native) | CUDA (NVIDIA), MPS (Apple — limited) |
| **Memory Model** | Unified Memory (CPU & GPU share same RAM) | Separate VRAM (GPU) + RAM (CPU), explicit transfers |
| **Data Transfer** | Zero-copy between CPU/GPU (same memory) | `tensor.to("cuda")` / `tensor.cpu()` copies required |
| **Graph Compilation** | `mx.compile()` for JIT optimization | `torch.compile()` (TorchDynamo + Inductor) |
| **Autodiff** | Functional: `mx.grad(fn)` returns a new function | OOP: `loss.backward()` mutates `.grad` attributes |
| **NumPy Interop** | Zero-copy `mx.array(np_array)` | `torch.from_numpy()` (shared memory, but device-bound) |
| **Ecosystem** | Growing (mlx-lm, mlx-data) | Massive (HuggingFace, torchvision, etc.) |

💡 **Key insight:** MLX's lazy evaluation + unified memory means the framework can fuse multiple operations into a single Metal kernel, avoiding intermediate memory allocations. This is a huge win on memory-bandwidth-bound workloads like LLM inference.

🎯 **Interview tip:** When asked "Why MLX over PyTorch on Mac?", lead with: *"Unified memory eliminates PCIe transfer overhead, and lazy evaluation enables kernel fusion — both critical for memory-bandwidth-bound LLM inference."*

---
## 📦 MLX Array Creation

MLX arrays are the fundamental data structure — similar to NumPy `ndarray` or PyTorch `Tensor`, but designed for lazy evaluation on Metal. Let's create arrays three ways: from Python lists, from NumPy (zero-copy!), and using random generators.

### From Python Lists

The simplest way to create an MLX array. MLX infers the dtype from the Python values.

In [ ]:
import mlx.core as mx

# 1-D array from a flat list
a = mx.array([1.0, 2.0, 3.0, 4.0])
print(f"1-D array: {a}")
print(f"  shape: {a.shape}")   # (4,)
print(f"  dtype: {a.dtype}")   # float32

# 2-D array from nested lists — think of it as (rows, cols)
b = mx.array([[1, 2, 3],
              [4, 5, 6]])      # shape: (2, 3)
print(f"\n2-D array:\n{b}")
print(f"  shape: {b.shape}")   # (2, 3)
print(f"  dtype: {b.dtype}")   # int32

# Scalar
s = mx.array(3.14)
print(f"\nScalar: {s}, shape: {s.shape}, dtype: {s.dtype}")

### From NumPy (Zero-Copy)

Because Apple Silicon uses unified memory, MLX can wrap a NumPy array without copying the data. This is a major advantage over PyTorch on NVIDIA, where you'd need an explicit `tensor.to("cuda")` copy across PCIe.

In [ ]:
import numpy as np

# Create a NumPy array
np_arr = np.random.randn(3, 4).astype(np.float32)  # shape: (3, 4)
print(f"NumPy array shape: {np_arr.shape}, dtype: {np_arr.dtype}")

# Zero-copy conversion to MLX
mlx_arr = mx.array(np_arr)  # No data copy on Apple Silicon!
print(f"MLX array shape:   {mlx_arr.shape}, dtype: {mlx_arr.dtype}")

# Verify values match
mx.eval(mlx_arr)
print(f"\nNumPy values:\n{np_arr}")
print(f"\nMLX values:\n{mlx_arr}")
print(f"\n💡 Same data, zero copy — unified memory at work!")

### Random Arrays

MLX provides random generators similar to NumPy. These are essential for weight initialization.

In [ ]:
# Uniform random: values in [0, 1)
uniform = mx.random.uniform(shape=(2, 3))  # shape: (2, 3)
print(f"Uniform [0,1):\n{uniform}")
print(f"  shape: {uniform.shape}, dtype: {uniform.dtype}\n")

# Normal (Gaussian) random: mean=0, std=1
normal = mx.random.normal(shape=(2, 3))    # shape: (2, 3)
print(f"Normal (μ=0, σ=1):\n{normal}")
print(f"  shape: {normal.shape}, dtype: {normal.dtype}\n")

# Random integers
randint = mx.random.randint(0, 100, shape=(2, 3))  # shape: (2, 3)
print(f"Random ints [0, 100):\n{randint}")
print(f"  shape: {randint.shape}, dtype: {randint.dtype}")

### Key Dtypes: float32, float16, bfloat16

Choosing the right dtype is critical for LLM training and inference. Smaller dtypes use less memory and bandwidth, enabling faster inference — but at the cost of precision.

| Dtype | Bytes | Range | Use Case |
|---|---|---|---|
| `float32` | 4 | ±3.4×10³⁸ | Full precision training, debugging |
| `float16` | 2 | ±65504 | Inference, mixed-precision training |
| `bfloat16` | 2 | ±3.4×10³⁸ | Training (same range as float32, less precision) |

⚡ **Performance tip:** `bfloat16` has the same exponent range as `float32` but fewer mantissa bits. This makes it more numerically stable for training than `float16`, which can overflow at values > 65504.

In [ ]:
# Compare dtypes: same data, different memory footprints
data = mx.random.normal(shape=(1000, 1000))  # shape: (1000, 1000)

f32 = data.astype(mx.float32)
f16 = data.astype(mx.float16)
bf16 = data.astype(mx.bfloat16)

print("Dtype comparison for a (1000, 1000) array:")
print(f"  float32  — {f32.dtype}, {f32.nbytes:>10,} bytes ({f32.nbytes / 1e6:.1f} MB)")
print(f"  float16  — {f16.dtype}, {f16.nbytes:>10,} bytes ({f16.nbytes / 1e6:.1f} MB)")
print(f"  bfloat16 — {bf16.dtype}, {bf16.nbytes:>10,} bytes ({bf16.nbytes / 1e6:.1f} MB)")
print(f"\n💡 float16 and bfloat16 use exactly half the memory of float32.")
print(f"   For a 7B parameter model: float32 = 28 GB, float16 = 14 GB, 4-bit = 3.5 GB")

---
## ⏳ Lazy Evaluation

This is the single most important concept in MLX. Unlike PyTorch (eager execution), MLX records operations into a computation graph and only executes them when you explicitly call `mx.eval()`. This enables **kernel fusion** — combining multiple operations into a single GPU dispatch.

### Operations Are Recorded, Not Executed

When you write `c = a + b`, MLX doesn't compute anything yet. It just records "add a and b" in a graph. The actual Metal GPU work happens only when you call `mx.eval(c)` or when you need the concrete value (e.g., `print(c)` or `c.item()`).

In [ ]:
a = mx.ones((1000, 1000))   # shape: (1000, 1000) — NOT computed yet
b = mx.ones((1000, 1000))   # shape: (1000, 1000) — NOT computed yet

# These operations are just recorded in the graph
c = a + b          # Recorded: add
d = c * 2          # Recorded: multiply
e = mx.sum(d)      # Recorded: reduce-sum

# Nothing has been computed on the GPU yet!
# The array 'e' exists as a node in the computation graph.
print(f"Type of e: {type(e)}")
print(f"Shape of e: {e.shape}")
print(f"Dtype of e: {e.dtype}")
print("(No Metal GPU work has happened yet — all lazy!)")

### Triggering Evaluation with `mx.eval()`

Now let's force the computation. When we call `mx.eval()`, MLX looks at the entire graph, fuses compatible operations, and dispatches them to the Metal GPU in one shot.

In [ ]:
# NOW we trigger computation
mx.eval(e)

print(f"Result: {e.item()}")
print(f"Expected: (1+1) * 2 * 1000 * 1000 = {(1+1) * 2 * 1000 * 1000}")
print(f"\n✅ The entire graph (add → multiply → sum) was fused and executed in one Metal dispatch.")

### Benchmark: Lazy vs Forced-Eager

Let's measure the performance difference. In "forced-eager" mode, we call `mx.eval()` after every single operation, preventing kernel fusion. In lazy mode, we let MLX fuse everything.

In [ ]:
import time

size = (2000, 2000)
n_trials = 10

# --- Forced-eager: eval after every op ---
eager_times = []
for _ in range(n_trials):
    x = mx.random.normal(shape=size)  # shape: (2000, 2000)
    mx.eval(x)
    t0 = time.perf_counter()
    y = x + x;       mx.eval(y)   # Force eval after add
    y = y * 0.5;     mx.eval(y)   # Force eval after multiply
    y = mx.exp(y);   mx.eval(y)   # Force eval after exp
    y = mx.sum(y);   mx.eval(y)   # Force eval after sum
    t1 = time.perf_counter()
    eager_times.append((t1 - t0) * 1000)

# --- Lazy: eval only at the end ---
lazy_times = []
for _ in range(n_trials):
    x = mx.random.normal(shape=size)  # shape: (2000, 2000)
    mx.eval(x)
    t0 = time.perf_counter()
    y = x + x                         # Lazy: recorded
    y = y * 0.5                        # Lazy: recorded
    y = mx.exp(y)                      # Lazy: recorded
    y = mx.sum(y)                      # Lazy: recorded
    mx.eval(y)                         # Single fused dispatch
    t1 = time.perf_counter()
    lazy_times.append((t1 - t0) * 1000)

eager_avg = sum(eager_times) / len(eager_times)
lazy_avg = sum(lazy_times) / len(lazy_times)
speedup = eager_avg / lazy_avg if lazy_avg > 0 else float('inf')

print(f"Forced-eager (eval after each op): {eager_avg:.2f} ms avg")
print(f"Lazy (single eval at end):         {lazy_avg:.2f} ms avg")
print(f"Speedup:                           {speedup:.1f}x")
print(f"\n⚡ Lazy evaluation lets MLX fuse add→mul→exp→sum into fewer Metal dispatches.")

### 💡 Kernel Fusion: Why Lazy Evaluation Matters

When MLX sees a chain of operations like `sum(exp(x * 0.5 + x))`, it can **fuse** them into a single Metal compute kernel instead of launching 4 separate kernels. This matters because:

1. **Fewer GPU dispatches** — each kernel launch has overhead (~5-20μs). Fusing N ops into 1 eliminates N-1 launches.
2. **No intermediate memory** — without fusion, each op writes its result to memory and the next op reads it back. Fused kernels keep intermediate values in fast GPU registers.
3. **Better memory bandwidth utilization** — the data is read once from memory, all operations applied, and the result written once.

This is especially impactful for LLM inference, which is **memory-bandwidth-bound**. Every avoided memory read/write directly translates to faster token generation.

⚠️ **Pitfall:** Don't call `mx.eval()` inside tight loops unless you need the value. Each `eval()` forces a graph break and prevents fusion across that boundary.

🎯 **Interview tip:** *"MLX's lazy evaluation enables kernel fusion, which reduces memory traffic — the primary bottleneck in LLM inference on Apple Silicon where compute is cheap but bandwidth is limited."*

---
## 🧮 Automatic Differentiation

MLX uses a **functional** approach to autodiff. Instead of calling `loss.backward()` like PyTorch, you create a gradient function with `mx.grad(fn)` that returns a *new function* computing the gradient. This is cleaner and more composable.

### `mx.grad` on Simple Functions

Let's start with functions we can verify by hand: f(x) = x² (derivative = 2x) and f(x) = sin(x) (derivative = cos(x)).

In [ ]:
# f(x) = x²  →  f'(x) = 2x
def f_square(x):
    return x * x

grad_square = mx.grad(f_square)  # Returns a NEW function that computes df/dx

x = mx.array(3.0)
print(f"f(x) = x²")
print(f"  f(3.0)  = {f_square(x).item():.1f}")       # 9.0
print(f"  f'(3.0) = {grad_square(x).item():.1f}")     # 6.0  (2 * 3)

# f(x) = sin(x)  →  f'(x) = cos(x)
def f_sin(x):
    return mx.sin(x)

grad_sin = mx.grad(f_sin)

x = mx.array(0.0)
print(f"\nf(x) = sin(x)")
print(f"  f(0.0)  = {f_sin(x).item():.4f}")           # 0.0
print(f"  f'(0.0) = {grad_sin(x).item():.4f}")        # 1.0  (cos(0) = 1)

x = mx.array(3.14159 / 2)  # π/2
print(f"\n  f(π/2)  = {f_sin(x).item():.4f}")          # 1.0
print(f"  f'(π/2) = {grad_sin(x).item():.4f}")         # ~0.0 (cos(π/2) ≈ 0)

### Second Derivatives

Since `mx.grad` returns a function, you can take the gradient of a gradient — giving you second derivatives. For f(x) = x², the second derivative is f''(x) = 2 (constant).

In [ ]:
# Second derivative: f(x) = x²  →  f'(x) = 2x  →  f''(x) = 2
first_deriv = mx.grad(f_square)          # f'(x) = 2x
second_deriv = mx.grad(first_deriv)      # f''(x) = 2

x = mx.array(5.0)
print(f"f(x) = x²")
print(f"  f(5)   = {f_square(x).item():.1f}")         # 25.0
print(f"  f'(5)  = {first_deriv(x).item():.1f}")      # 10.0
print(f"  f''(5) = {second_deriv(x).item():.1f}")      # 2.0

# Second derivative of sin: f''(x) = -sin(x)
sin_second = mx.grad(mx.grad(f_sin))
x = mx.array(3.14159 / 2)  # π/2
print(f"\nf(x) = sin(x)")
print(f"  f''(π/2) = {sin_second(x).item():.4f}")      # -sin(π/2) = -1.0

### `mx.grad` on MSE Loss Function

In practice, we differentiate loss functions. Let's compute the gradient of Mean Squared Error (MSE) — the loss used in regression tasks. MSE = mean((predictions - targets)²).

In [ ]:
import numpy as np

# MSE loss: L(pred) = mean((pred - target)²)
# Gradient: dL/dpred = 2 * (pred - target) / n

def mse_loss(predictions, targets):
    """Mean Squared Error loss."""
    diff = predictions - targets                # shape: (n,)
    return mx.mean(diff * diff)                 # shape: scalar

# Create sample data
targets = mx.array([1.0, 2.0, 3.0, 4.0])      # shape: (4,)
predictions = mx.array([1.5, 2.5, 2.5, 3.5])   # shape: (4,)

# Compute loss
loss = mse_loss(predictions, targets)
print(f"Predictions: {predictions}")
print(f"Targets:     {targets}")
print(f"MSE Loss:    {loss.item():.4f}")

# Compute gradient of loss w.r.t. predictions
grad_fn = mx.grad(mse_loss)  # Gradient w.r.t. first argument by default
grads = grad_fn(predictions, targets)           # shape: (4,)
print(f"\nGradients (dL/dpred): {grads}")
print(f"Expected: 2*(pred-target)/n = {2 * (np.array([1.5,2.5,2.5,3.5]) - np.array([1,2,3,4])) / 4}")
print(f"\n💡 Positive gradient → prediction too high, negative → too low.")

### Gradient Visualization

Let's visualize how the gradient of f(x) = x² changes across different input values. The gradient tells us the slope of the function at each point — the direction and magnitude of steepest ascent.

In [ ]:
import matplotlib.pyplot as plt

# Compute f(x) and f'(x) over a range
x_vals = np.linspace(-3, 3, 100)
y_vals = x_vals ** 2                    # f(x) = x²
grad_vals = 2 * x_vals                  # f'(x) = 2x

# Verify with MLX grad at a few points
mlx_grads = []
grad_fn = mx.grad(f_square)
for xv in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    g = grad_fn(mx.array(xv)).item()
    mlx_grads.append((xv, g))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: function and tangent lines
ax1.plot(x_vals, y_vals, 'b-', linewidth=2, label='f(x) = x²')
for xv, gv in mlx_grads:
    yv = xv ** 2
    tangent_x = np.linspace(xv - 0.8, xv + 0.8, 20)
    tangent_y = gv * (tangent_x - xv) + yv
    ax1.plot(tangent_x, tangent_y, '--', alpha=0.7, linewidth=1.5)
    ax1.plot(xv, yv, 'ro', markersize=6)
ax1.set_title('f(x) = x² with tangent lines')
ax1.set_xlabel('x')
ax1.set_ylabel('f(x)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-1, 10)

# Right: gradient values
ax2.plot(x_vals, grad_vals, 'r-', linewidth=2, label="f'(x) = 2x")
for xv, gv in mlx_grads:
    ax2.plot(xv, gv, 'ko', markersize=8)
    ax2.annotate(f'  mx.grad → {gv:.1f}', (xv, gv), fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax2.set_title("Gradient f'(x) = 2x (computed by mx.grad)")
ax2.set_xlabel('x')
ax2.set_ylabel("f'(x)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("🎯 The gradient points in the direction of steepest ascent. Training minimizes loss by going OPPOSITE to the gradient.")

---
## 🧱 `mlx.nn` Modules

MLX provides a `nn` module similar to PyTorch's `torch.nn`. These are the building blocks we'll use to construct transformers. Let's explore the key layers.

### Linear Layer

A linear layer computes `y = xW^T + b`. It's the most fundamental neural network operation — every transformer uses dozens of them for Q/K/V projections, FFN layers, and output heads.

In [ ]:
import mlx.nn as nn

# Create a linear layer: 4 input features → 3 output features
linear = nn.Linear(4, 3)

# Inspect parameters
print("Linear(4, 3) parameters:")
print(f"  weight shape: {linear.weight.shape}")   # (3, 4) — note: (out, in)
print(f"  weight dtype: {linear.weight.dtype}")
print(f"  bias shape:   {linear.bias.shape}")      # (3,)
print(f"  bias dtype:   {linear.bias.dtype}")

# Forward pass
x = mx.random.normal(shape=(2, 4))  # shape: (batch=2, in_features=4)
y = linear(x)                        # shape: (batch=2, out_features=3)
mx.eval(y)

print(f"\nForward pass:")
print(f"  Input  shape: {x.shape}")    # (2, 4)
print(f"  Output shape: {y.shape}")    # (2, 3)
print(f"  Output:\n{y}")

# Count parameters
n_params = linear.weight.size + linear.bias.size
print(f"\n  Total parameters: {n_params} (weight: {linear.weight.size} + bias: {linear.bias.size})")

### Embedding Layer

An embedding layer is a lookup table that maps integer token IDs to dense vectors. It's the first layer in every language model — converting discrete tokens into continuous representations the model can process.

In [ ]:
# Embedding: vocab_size=100 tokens, each mapped to a 32-dim vector
embedding = nn.Embedding(num_embeddings=100, dims=32)

print(f"Embedding(100, 32) parameters:")
print(f"  weight shape: {embedding.weight.shape}")  # (100, 32) — lookup table
print(f"  Total params: {embedding.weight.size}")    # 100 * 32 = 3200

# Look up embeddings for token IDs
token_ids = mx.array([5, 42, 99, 0])  # shape: (4,) — 4 token IDs
embeddings = embedding(token_ids)       # shape: (4, 32) — 4 embedding vectors
mx.eval(embeddings)

print(f"\nForward pass:")
print(f"  Token IDs shape:  {token_ids.shape}")    # (4,)
print(f"  Embeddings shape: {embeddings.shape}")    # (4, 32)
print(f"  First embedding (token 5): [{embeddings[0, :4].tolist()}... ]  (showing first 4 dims)")

# Batch of sequences
batch_ids = mx.array([[1, 2, 3],
                       [4, 5, 6]])      # shape: (batch=2, seq_len=3)
batch_emb = embedding(batch_ids)         # shape: (2, 3, 32)
mx.eval(batch_emb)
print(f"\nBatch forward pass:")
print(f"  Input shape:  {batch_ids.shape}")   # (2, 3)
print(f"  Output shape: {batch_emb.shape}")   # (2, 3, 32)

### LayerNorm and RMSNorm

Normalization layers stabilize training by keeping activations in a reasonable range. Two variants matter for transformers:

- **LayerNorm**: Normalizes to zero mean and unit variance, then applies learnable scale (γ) and shift (β). Used in original Transformer, GPT-2.
- **RMSNorm**: Only normalizes by the root-mean-square (no mean subtraction, no shift). Simpler and faster. Used in LLaMA, Gemma, Mistral.

💡 **Why RMSNorm won:** It's ~10-15% faster than LayerNorm with negligible quality difference. Modern LLMs almost universally use RMSNorm.

In [ ]:
# LayerNorm: normalizes across the last dimension
layer_norm = nn.LayerNorm(dims=32)

x = mx.random.normal(shape=(2, 5, 32))  # shape: (batch=2, seq_len=5, d_model=32)
ln_out = layer_norm(x)                    # shape: (2, 5, 32) — same shape
mx.eval(ln_out)

print("LayerNorm(32):")
print(f"  Input shape:  {x.shape}")       # (2, 5, 32)
print(f"  Output shape: {ln_out.shape}")   # (2, 5, 32)
print(f"  Parameters: weight {layer_norm.weight.shape}, bias {layer_norm.bias.shape}")

# Verify: output should have ~zero mean and ~unit variance along last dim
mean = mx.mean(ln_out, axis=-1)
var = mx.var(ln_out, axis=-1)
mx.eval(mean, var)
print(f"  Output mean (should be ~0): {mean[0, 0].item():.6f}")
print(f"  Output var  (should be ~1): {var[0, 0].item():.6f}")

print()

# RMSNorm: simpler — no mean subtraction, no bias
rms_norm = nn.RMSNorm(dims=32)

rms_out = rms_norm(x)                     # shape: (2, 5, 32) — same shape
mx.eval(rms_out)

print("RMSNorm(32):")
print(f"  Input shape:  {x.shape}")        # (2, 5, 32)
print(f"  Output shape: {rms_out.shape}")   # (2, 5, 32)
print(f"  Parameters: weight {rms_norm.weight.shape} (no bias!)")

# RMSNorm output has unit RMS (root-mean-square)
rms = mx.sqrt(mx.mean(rms_out * rms_out, axis=-1))
mx.eval(rms)
print(f"  Output RMS (should be ~1): {rms[0, 0].item():.4f}")

### Activation Functions: ReLU, GELU, SiLU

Activation functions introduce non-linearity. Without them, stacking linear layers would just produce another linear function. Here are the three you'll see in modern LLMs:

- **ReLU**: max(0, x) — simple, fast, but "dead neurons" problem (gradient = 0 for x < 0)
- **GELU**: x · Φ(x) — smooth approximation of ReLU, used in GPT-2, BERT
- **SiLU** (Swish): x · σ(x) — used in LLaMA's SwiGLU FFN, smooth and self-gated

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Compute activations over a range
x_np = np.linspace(-4, 4, 200)
x_mx = mx.array(x_np.astype(np.float32))  # shape: (200,)

relu_out = nn.relu(x_mx)
gelu_out = nn.gelu(x_mx)
silu_out = nn.silu(x_mx)
mx.eval(relu_out, gelu_out, silu_out)

# Convert to numpy for plotting
relu_np = np.array(relu_out)
gelu_np = np.array(gelu_out)
silu_np = np.array(silu_out)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, name, vals, color in [
    (axes[0], "ReLU", relu_np, "tab:blue"),
    (axes[1], "GELU", gelu_np, "tab:orange"),
    (axes[2], "SiLU (Swish)", silu_np, "tab:green"),
]:
    ax.plot(x_np, vals, color=color, linewidth=2, label=name)
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
    ax.set_title(name)
    ax.set_xlabel('x')
    ax.set_ylabel('activation(x)')
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_ylim(-1.5, 4.5)

plt.tight_layout()
plt.show()

print("💡 GELU and SiLU are smooth — they allow small negative values through,")
print("   which helps gradient flow compared to ReLU's hard cutoff at 0.")
print("🎯 Interview: LLaMA uses SiLU in its SwiGLU FFN. GPT-2/BERT use GELU.")

---
## 🔄 MLX Training Loop Pattern

Now let's put it all together. The MLX training loop has a distinctive pattern that differs from PyTorch:

1. Define model (subclass `nn.Module`)
2. Create loss function
3. Wrap with `nn.value_and_grad()` — returns both loss AND gradients in one call
4. Call `optimizer.update(model, grads)` — applies gradients to parameters
5. Call `mx.eval()` — triggers actual Metal GPU computation

This is the pattern you'll use for every training task in this series.

### Step 1: Define TinyMLP

A simple 2-layer MLP (Multi-Layer Perceptron). Input → Hidden (with ReLU) → Output. We'll train it to learn the function y = sum(x).

In [ ]:
import mlx.optimizers as optim

class TinyMLP(nn.Module):
    """2-layer MLP: input(4) → hidden(32) → output(1)."""
    
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 32)    # 4 inputs → 32 hidden
        self.layer2 = nn.Linear(32, 1)    # 32 hidden → 1 output
    
    def __call__(self, x):
        # x shape: (batch, 4)
        h = nn.relu(self.layer1(x))       # shape: (batch, 32)
        out = self.layer2(h)              # shape: (batch, 1)
        return out

model = TinyMLP()

# Count parameters by flattening the nested parameter tree
def count_params(params):
    total = 0
    for v in params.values():
        if isinstance(v, dict):
            total += count_params(v)
        elif hasattr(v, 'size'):
            total += v.size
    return total

param_count = count_params(model.parameters())

print(f"TinyMLP architecture:")
print(f"  Layer 1: Linear(4, 32)  — weight {model.layer1.weight.shape}, bias {model.layer1.bias.shape}")
print(f"  Layer 2: Linear(32, 1)  — weight {model.layer2.weight.shape}, bias {model.layer2.bias.shape}")
print(f"  Total parameters: {param_count}")
print(f"  Task: learn y = sum(x) where x is a 4-dim vector")

### Step 2: `nn.value_and_grad()` Setup

This is the key MLX pattern. `nn.value_and_grad(model, loss_fn)` returns a function that computes both the loss value and the gradients in a single call — no separate `.backward()` needed.

In [ ]:
# Define loss function: MSE between model output and target
def loss_fn(model, x, y):
    pred = model(x)                       # shape: (batch, 1)
    return mx.mean((pred - y) ** 2)       # scalar MSE loss

# Create the combined loss+gradient function
# This returns (loss_value, gradients) in one call
loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

# Set up optimizer
optimizer = optim.Adam(learning_rate=1e-3)

print("Training setup:")
print(f"  Loss function: MSE")
print(f"  Optimizer: Adam (lr=1e-3)")
print(f"  loss_and_grad_fn returns: (loss_value, parameter_gradients)")
print(f"\n💡 In PyTorch you'd do: loss.backward() then optimizer.step()")
print(f"   In MLX: loss, grads = loss_and_grad_fn(model, x, y)")

### Step 3: `optimizer.update()` and `mx.eval()` Pattern

Now we train the model to learn y = sum(x). Each step:
1. Generate a random batch of x vectors
2. Compute target y = sum(x)
3. Get loss and gradients via `loss_and_grad_fn`
4. Update parameters via `optimizer.update`
5. Force evaluation with `mx.eval` (triggers Metal GPU work)

In [ ]:
import time

# Re-initialize for a clean training run
model = TinyMLP()
optimizer = optim.Adam(learning_rate=1e-3)
loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

losses = []
n_steps = 500
batch_size = 64

t_start = time.perf_counter()

for step in range(n_steps):
    # Generate random training data: y = sum(x)
    x = mx.random.normal(shape=(batch_size, 4))   # shape: (64, 4)
    y = mx.sum(x, axis=1, keepdims=True)           # shape: (64, 1)
    
    # Forward + backward (both lazy)
    loss, grads = loss_and_grad_fn(model, x, y)
    
    # Update parameters (lazy)
    optimizer.update(model, grads)
    
    # Force evaluation — this is where Metal GPU actually computes
    mx.eval(model.parameters(), optimizer.state, loss)
    
    loss_val = loss.item()
    losses.append(loss_val)
    
    if step % 100 == 0 or step == n_steps - 1:
        print(f"  Step {step:4d} | Loss: {loss_val:.6f}")

t_end = time.perf_counter()
print(f"\nTraining complete in {t_end - t_start:.2f}s ({n_steps} steps)")
print(f"Final loss: {losses[-1]:.6f}")

### Training Loss Curve

Let's visualize how the loss decreased during training using our shared `utils.viz.plot_loss_curve()` helper.

In [ ]:
from utils.viz import plot_loss_curve

fig = plot_loss_curve(losses, title="TinyMLP Training: y = sum(x)")
plt.show()

# Test the trained model
print("Testing trained model:")
test_x = mx.array([[1.0, 2.0, 3.0, 4.0]])   # shape: (1, 4), expected sum = 10.0
pred = model(test_x)                           # shape: (1, 1)
mx.eval(pred)
print(f"  Input:    {test_x.tolist()[0]}")
print(f"  Expected: {sum([1.0, 2.0, 3.0, 4.0])}")
print(f"  Predicted: {pred.item():.4f}")

test_x2 = mx.array([[-1.0, 0.5, 2.0, -0.5]])  # shape: (1, 4), expected sum = 1.0
pred2 = model(test_x2)
mx.eval(pred2)
print(f"\n  Input:    {test_x2.tolist()[0]}")
print(f"  Expected: {sum([-1.0, 0.5, 2.0, -0.5])}")
print(f"  Predicted: {pred2.item():.4f}")

print(f"\n✅ The model learned y = sum(x) from random data!")

### ⚠️ Common Pitfalls & Tips

**⚠️ Pitfall: Forgetting `mx.eval()` in the training loop.**
Without it, the computation graph grows unboundedly and you'll run out of memory. Always call `mx.eval()` at the end of each training step.

**⚠️ Pitfall: Calling `mx.eval()` too often.**
Each `eval()` forces a graph break. If you eval inside the loss function, you prevent fusion between the forward pass and gradient computation.

**💡 Insight: The MLX training pattern.**
```python
loss, grads = loss_and_grad_fn(model, x, y)  # Lazy: builds graph
optimizer.update(model, grads)                 # Lazy: records updates
mx.eval(model.parameters(), optimizer.state, loss)  # Execute everything
```
This three-line pattern is the core of every MLX training loop. Memorize it.

**🎯 Interview tip:** *"MLX's `nn.value_and_grad` is functional — it returns a new function that computes both the loss and gradients. Combined with lazy evaluation, the entire forward-backward-update cycle can be fused into an optimized Metal kernel graph."*

---
**Next up:** Notebook 02 — Math Foundations (dot products, softmax, cross-entropy loss)

## 📜 History & Alternatives

### The Evolution of ML Frameworks

Every generation of ML frameworks solved a key limitation of its predecessor. The trajectory shows a clear trend: from manual computation graphs → automatic differentiation → hardware-specific optimization.

| Year | Framework | Team | Key Contribution |
|------|-----------|------|-----------------|
| 2007 | **Theano** | Mila (Bengio lab) | First symbolic tensor compiler with automatic differentiation |
| 2015 | **Caffe** | Berkeley AI Research | Fast CNN inference, model zoo concept, production deployment |
| 2015 | **TensorFlow 1.x** | Google Brain | Static computation graphs, distributed training, TF Serving |
| 2016 | **PyTorch** | Meta AI (FAIR) | Dynamic computation graphs (define-by-run), Pythonic API — became research standard |
| 2018 | **JAX** | Google DeepMind | Functional transforms (jit, grad, vmap, pmap) on top of XLA compiler |
| 2019 | TensorFlow 2.x | Google Brain | Eager execution by default, Keras integration — caught up to PyTorch UX |
| 2020 | **Hugging Face Transformers** | Hugging Face | Model hub + unified API — democratized access to pretrained models |
| 2022 | **tinygrad** | George Hotz | Minimalist framework (~5000 LOC), educational, multi-backend |
| 2023 | **MLX** | Apple ML Research | Designed for Apple Silicon UMA — lazy evaluation, unified memory, NumPy-like API |
| 2024 | MLX-LM / mlx-vlm | Apple + Community | High-level LLM/VLM serving built on MLX, quantization support |

### 💡 Why MLX Exists

MLX fills a specific niche: **high-performance ML on Apple Silicon**. While PyTorch and JAX support MPS (Metal Performance Shaders), they treat Apple GPUs as a secondary target. MLX is built from the ground up for unified memory:

- **Lazy evaluation**: Operations are recorded but not executed until results are needed — enables graph-level optimization
- **Unified memory**: Arrays live in shared memory accessible by CPU and GPU — zero-copy transfers
- **Composable transforms**: `mx.grad()`, `mx.vmap()`, `mx.compile()` — inspired by JAX's functional approach
- **NumPy-like API**: Minimal learning curve for Python ML practitioners

### Framework Comparison for Apple Silicon

| Feature | PyTorch (MPS) | JAX (Metal) | MLX |
|---------|--------------|-------------|-----|
| Apple Silicon optimization | Secondary | Experimental | Primary |
| Unified memory | Partial | No | Native |
| Lazy evaluation | No (eager) | Yes (via XLA) | Yes |
| Community size | Massive | Large | Growing |
| Model availability | Extensive | Good | Growing (mlx-community) |
| Training support | Full | Full | Good (improving) |

### ⚡ Performance Note

For inference on Apple Silicon, MLX typically outperforms PyTorch MPS by 1.5-3× on transformer models due to its native unified memory support and Metal-optimized kernels. The gap is largest for memory-bandwidth-bound operations like large matrix multiplications in LLM decoding.

### 🎯 Interview Tip

> *"MLX's key design choices — lazy evaluation, unified memory, and composable function transforms — mirror JAX's philosophy but are purpose-built for Apple Silicon's UMA. The lazy evaluation model allows MLX to fuse operations and optimize memory access patterns before execution, which is critical when memory bandwidth (not compute) is the bottleneck for LLM inference."*